# 01 — Data Ingestion & Preparation

This notebook builds the complete input dataset used by all subsequent notebooks.

**Pipeline overview:**
1. Download PM2.5 daily and hourly data from GIOŚ for all stations
2. Filter stations by completeness criteria (2016–2019)
3. Download meteorological data from Open-Meteo for each station
4. Load station metadata (GIOŚ) and enrich with OSMnx features
5. Extract Copernicus GHSL raster features (population, built-up surface, volume)
6. Merge all sources into final flat CSVs

**Output files:**
- `stations_complete_2016_2019.csv` — PM2.5 24h, filtered stations
- `stations_complete_2016_2019_1h.csv` — PM2.5 1h, filtered stations
- `stations_weather_2016_2019.csv` — PM2.5 24h + Open-Meteo
- `stations_weather_2016_2019_1h.csv` — PM2.5 1h + Open-Meteo
- `metaall.csv` — station metadata with all spatial features
- `all_data.csv` — final merged dataset (24h)
- `all_data_1h.csv` — final merged dataset (1h)

## 0. Imports & configuration

In [9]:
import os
import time
import warnings

import numpy as np
import pandas as pd
import xarray as xr
import rasterio
from rasterio.merge import merge
from rasterio.mask import mask
import geopandas as gpd
from shapely.geometry import Point
import osmnx as ox

from read_data import read_data as read_data_daily
from read_data_hour import read_data as read_data_hourly
from read_openmeteo import read_openmeteo
from read_openmeteo_hour import read_openmeteo as read_openmeteo_hour

warnings.filterwarnings("ignore")
ox.settings.use_cache = True
ox.settings.log_console = False

# === GLOBAL PARAMETERS ===
START_YEAR = 2016
END_YEAR   = 2019

# Completeness filter thresholds (Vovk et al. 2024)
MIN_YEARS           = 3
MIN_MONTHS_PER_YEAR = 12
MIN_DAYS_PER_MONTH  = 14

# Spatial buffer around each station (meters)
BUFFER_RADIUS_M = 5000

# Open-Meteo API: pause between requests (seconds)
API_SLEEP_S = 15

print("Configuration loaded.")
print(f"  Period : {START_YEAR}–{END_YEAR}")
print(f"  Buffer : {BUFFER_RADIUS_M} m")

Configuration loaded.
  Period : 2016–2019
  Buffer : 5000 m


## 1. PM2.5 data — download and completeness filter

Source: Chief Inspectorate of Environmental Protection (GIOŚ)  
https://powietrze.gios.gov.pl/pjp/archives

Completeness criterion: station must have ≥ `MIN_YEARS` years, each with
≥ `MIN_MONTHS_PER_YEAR` months that each contain ≥ `MIN_DAYS_PER_MONTH` valid days.

In [10]:
def is_station_complete(series: pd.Series) -> bool:
    """Return True if the station meets the completeness criteria."""
    s = series.dropna()
    if s.empty:
        return False

    counts = s.groupby([s.index.year, s.index.month]).size()

    valid_years = 0
    for year in range(START_YEAR, END_YEAR + 1):
        if year not in counts.index.get_level_values(0):
            continue
        days_per_month = counts.loc[year]
        if (days_per_month >= MIN_DAYS_PER_MONTH).sum() >= MIN_MONTHS_PER_YEAR:
            valid_years += 1

    return valid_years >= MIN_YEARS


def load_and_filter_pm(resolution: str) -> pd.DataFrame:
    """
    Load PM2.5 data for a given resolution ('PM25_24g' or 'PM25_1g'),
    restrict to 2016–2019, and keep only stations meeting completeness criteria.
    """
    if resolution == "PM25_24g":
        df_merged, _ = read_data_daily(resolution)
    else:
        df_merged, _ = read_data_hourly(resolution)

    # Save merged NetCDF for potential reuse
    nc_path = f"{resolution}_merged.nc"
    df_merged.to_xarray().to_netcdf(nc_path)
    print(f"  Saved {nc_path}")

    # Restrict to study period
    df = df_merged.copy()
    df.index = pd.to_datetime(df.index)
    df.index.name = "date"
    df = df[(df.index.year >= START_YEAR) & (df.index.year <= END_YEAR)]

    # Filter stations
    kept = {col: df[col] for col in df.columns if is_station_complete(df[col])}
    df_clean = pd.DataFrame(kept).dropna(how="all")

    print(f"  {resolution}: kept {df_clean.shape[1]} / {df.shape[1]} stations")
    return df_clean


print("Loading 24h PM2.5...")
df_pm25_24h = load_and_filter_pm("PM25_24g")
df_pm25_24h.to_csv("stations_complete_2016_2019.csv")
print("  → stations_complete_2016_2019.csv")

print("\nLoading 1h PM2.5...")
df_pm25_1h = load_and_filter_pm("PM25_1g")
df_pm25_1h.to_csv("stations_complete_2016_2019_1h.csv")
print("  → stations_complete_2016_2019_1h.csv")

Loading 24h PM2.5...
Wczytywanie: .\2016\2016_PM25_24g.xlsx
Wczytywanie: .\2017\2017_PM25_24g.xlsx
Wczytywanie: .\2018\2018_PM25_24g.xlsx
Wczytywanie: .\2019\2019_PM25_24g.xlsx
            DsLegAlRzecz  DsOsieczow21  DsWalbrzWyso  DsWrocNaGrob  \
Data                                                                 
2016-01-01          59.3         19.00           NaN           NaN   
2016-01-02          54.7         48.50           NaN           NaN   
2016-01-03          69.0         54.90           NaN           NaN   
2016-01-04          80.1         78.40           NaN           NaN   
2016-01-05         100.5         79.50           NaN         80.40   
...                  ...           ...           ...           ...   
2019-12-27           NaN          7.43          7.34          7.00   
2019-12-28           NaN          7.75          7.64          5.74   
2019-12-29           NaN         14.62         11.79          9.29   
2019-12-30           NaN         12.79         13.46 

## 2. Station metadata — GIOŚ coordinates + OSMnx spatial features

OSMnx requires internet access to the Overpass API (OpenStreetMap), which may
not be available in all environments (e.g. restricted sandboxes).

**Workflow:**
1. This cell loads GIOŚ coordinates and saves `metadata_coords.csv`
2. Run `fetch_osm_features.py` **locally** to produce `metadata5000best.csv`
3. Place `metadata5000best.csv` next to this notebook and continue

Features extracted per station (5 km buffer):
- `nearest_road_dist_m` — distance to nearest road [m]
- `total_road_length_km` — total road network length [km]
- `building_count` — number of buildings

In [11]:
# Load GIOŚ metadata
df_meta_raw = pd.read_excel("meta.xlsx", decimal=",")
df_meta_raw.set_index("Nr", inplace=True, drop=True)
df_meta_raw.columns = (
    df_meta_raw.columns
    .str.strip()
    .str.lower()
    .str.replace(r"\s+", "_", regex=True)
)
df_meta_raw = df_meta_raw.rename(columns={
    "stary_kod_stacji_(o_ile_inny_od_aktualnego)": "stary_kod",
    "wgs84_φ_n": "lat",
    "wgs84_λ_e": "lon",
})
df_meta_raw.reset_index(inplace=True)
df_meta_raw.set_index("kod_stacji", inplace=True)

# Keep only stations present in filtered PM2.5 dataset
stations_in_data = df_pm25_24h.columns
df_meta = df_meta_raw.loc[
    df_meta_raw.index.intersection(stations_in_data), ["lat", "lon"]
].copy()
df_meta.index.name = "stacja"

print(f"Stations in metadata: {len(df_meta)}")
df_meta.head()


Stations in metadata: 61


,lat,lon
stacja,,
DsLegAlRzecz,51.204503,16.180513
DsOsieczow21,51.317630,15.431719
DsWalbrzWyso,50.768729,16.269677
DsWrocNaGrob,51.103456,17.059225
DsZgorBohGet,51.150391,15.008175


In [12]:
OSM_OUTPUT = "metadata5000best.csv"
OSM_COLS   = ["nearest_road_dist_m", "total_road_length_km", "building_count"]

def compute_osm_features(
    lat: float, lon: float, buffer_radius: int = BUFFER_RADIUS_M
) -> tuple:
    """
    Extract road and building features from OpenStreetMap within a
    circular buffer around the given coordinates.

    Returns
    -------
    (nearest_road_dist_m, total_road_length_km, building_count)
    All values are np.nan on failure.
    """
    try:
        pt          = Point(lon, lat)
        buffer_poly = ox.utils_geo.buffer_geometry(pt, buffer_radius)
        buffer_3857 = gpd.GeoSeries([buffer_poly], crs="EPSG:4326").to_crs(epsg=3857).iloc[0]
        pt_3857     = gpd.GeoSeries([pt],          crs="EPSG:4326").to_crs(epsg=3857).iloc[0]

        # Roads
        try:
            G = ox.graph_from_point((lat, lon), dist=buffer_radius, network_type="drive")
            _, edges = ox.graph_to_gdfs(G)
            edges    = edges.to_crs(epsg=3857)
            edges_in = edges[edges.geometry.intersects(buffer_3857)]
            total_road_length_km = edges_in.length.sum() / 1000 if not edges_in.empty else np.nan
            nearest_road_dist_m  = edges_in.distance(pt_3857).min()  if not edges_in.empty else np.nan
        except Exception as e:
            print(f"    [road error] ({lat}, {lon}): {e}")
            total_road_length_km = np.nan
            nearest_road_dist_m  = np.nan

        # Buildings
        try:
            buildings    = ox.features_from_polygon(buffer_poly, tags={"building": True})
            buildings    = buildings[
                buildings.geometry.geom_type.isin(["Polygon", "MultiPolygon"])
            ].to_crs(epsg=3857)
            buildings_in = buildings[buildings.geometry.intersects(buffer_3857)]
            building_count = len(buildings_in)
        except Exception as e:
            print(f"    [building error] ({lat}, {lon}): {e}")
            building_count = np.nan

        return nearest_road_dist_m, total_road_length_km, building_count

    except Exception as e:
        print(f"    [general error] ({lat}, {lon}): {e}")
        return np.nan, np.nan, np.nan


# ── Load from cache or compute ────────────────────────────────────────────────
if os.path.exists(OSM_OUTPUT):
    df_meta = pd.read_csv(OSM_OUTPUT, index_col=0)
    df_meta = df_meta.drop(columns=[c for c in df_meta.columns if "Unnamed" in c], errors="ignore")
    df_meta.index.name = "stacja"
    rows_todo = df_meta[df_meta[OSM_COLS].isna().any(axis=1)]
    print(f"Cache found: {OSM_OUTPUT} ({len(df_meta)} stations, {len(rows_todo)} with missing OSMnx features)")
else:
    df_meta[OSM_COLS[0]] = np.nan
    df_meta[OSM_COLS[1]] = np.nan
    df_meta[OSM_COLS[2]] = np.nan
    rows_todo = df_meta
    print(f"No cache. Will compute OSMnx features for {len(rows_todo)} stations.")

# Only compute rows that are still missing
for i, (station, row) in enumerate(rows_todo.iterrows(), 1):
    print(f"  [{i}/{len(rows_todo)}] {station}")
    nr, trl, bc = compute_osm_features(row["lat"], row["lon"])
    df_meta.at[station, "nearest_road_dist_m"]  = nr
    df_meta.at[station, "total_road_length_km"] = trl
    df_meta.at[station, "building_count"]        = bc

df_meta.to_csv(OSM_OUTPUT)

# ── Fill any remaining NaNs with column median ────────────────────────────────
nan_mask  = df_meta[OSM_COLS].isna().any(axis=1)
nan_count = nan_mask.sum()
if nan_count > 0:
    print(f"\nWARNING: {nan_count} station(s) still have missing OSMnx features — filling with column median.")
    for col in OSM_COLS:
        n_filled = df_meta[col].isna().sum()
        if n_filled > 0:
            df_meta[col] = df_meta[col].fillna(df_meta[col].median())
            print(f"  {col}: filled {n_filled} NaN(s) with median = {df_meta[col].median():.2f}")

print(f"\nSaved {OSM_OUTPUT} ({len(df_meta)} stations)")
df_meta.head()


Cache found: metadata5000best.csv (61 stations, 0 with missing OSMnx features)

Saved metadata5000best.csv (61 stations)


,lat,lon,nearest_road_dist_m,total_road_length_km,building_count
stacja,,,,,
0,51.204503,16.180513,98.613062,942.949585,15745.0
1,51.317630,15.431719,40.258760,148.954010,2334.0
2,50.768729,16.269677,170.100903,817.813995,15897.0
3,51.103456,17.059225,97.176639,1518.020860,60249.0
4,51.150391,15.008175,138.987771,988.524194,28363.0


In [13]:
import pandas as pd

# 1. Wczytujemy bazowe współrzędne z nazwami stacji
df_coords = pd.read_csv("metadata_coords.csv")

# 2. Wczytujemy dane z OSM (best)
df_osm = pd.read_csv("metadata5000best.csv")

# 3. Standaryzacja współrzędnych (zaokrąglenie do 6 miejsc po przecinku)
# Zapobiega to błędom łączenia wynikającym z minimalnych różnic w precyzji float
precision = 3
df_coords['lat_r'] = df_coords['lat'].round(precision)
df_coords['lon_r'] = df_coords['lon'].round(precision)
df_osm['lat_r'] = df_osm['lat'].round(precision)
df_osm['lon_r'] = df_osm['lon'].round(precision)

# 4. Merge: bierzemy stacje z df_coords i doklejamy do nich kolumny z df_osm
# Usuwamy kolumny 'stacja', 'lat' i 'lon' z df_osm, bo mamy je już w df_coords
df_meta = pd.merge(
    df_coords, 
    df_osm.drop(columns=['stacja', 'lat', 'lon']), 
    on=['lat_r', 'lon_r'], 
    how='left'
)

# 5. Czyszczenie: usuwamy kolumny pomocnicze i ustawiamy 'stacja' jako indeks
df_meta = df_meta.drop(columns=['lat_r', 'lon_r'])
df_meta = df_meta.set_index("stacja")

print(f"Metadata enriched: {len(df_meta)} stations")
print(df_meta.head())

Metadata enriched: 45 stations
                    lat        lon  nearest_road_dist_m  total_road_length_km  \
stacja                                                                          
DsOsieczow21  51.317630  15.431719            40.258760            148.954010   
DsWalbrzWyso  50.768729  16.269677           170.100903            817.813995   
DsWrocNaGrob  51.103456  17.059225            97.176639           1518.020860   
DsZgorBohGet  51.150391  15.008175           138.987771            988.524194   
KpBydFieldor  53.151452  18.132062            22.380118            711.014879   

              building_count  
stacja                        
DsOsieczow21          2334.0  
DsWalbrzWyso         15897.0  
DsWrocNaGrob         60249.0  
DsZgorBohGet         28363.0  
KpBydFieldor          8995.0  


## 3. Meteorological data — Open-Meteo API

Daily and hourly weather is fetched per station from the Open-Meteo archive API.
Elevation is also returned and stored in the metadata.

`read_openmeteo(lon, lat)` → daily DataFrame + elevation  
`read_openmeteo_hour(lon, lat)` → hourly DataFrame + elevation

In [14]:
def fetch_weather_all_stations(
    df_meta: pd.DataFrame,
    pm_wide: pd.DataFrame,
    fetch_fn,
    output_csv: str,
    resolution_label: str = "daily",
) -> pd.DataFrame:
    """
    For each station in df_meta, download weather data and merge with PM2.5.
    Progress is saved to output_csv after every station so the function
    can be safely interrupted and resumed (API rate-limit safe).

    Parameters
    ----------
    df_meta          : DataFrame with columns [lat, lon], indexed by station code
    pm_wide          : Wide PM2.5 DataFrame (rows = dates, columns = stations)
    fetch_fn         : read_openmeteo or read_openmeteo_hour
    output_csv       : path for the merged output CSV
    resolution_label : label for logging only

    Returns
    -------
    df_full : merged DataFrame (long format)
    """
    # Convert PM2.5 from wide to long
    pm_long = pm_wide.reset_index().melt(
        id_vars="date", var_name="stacja", value_name="pm25"
    )
    pm_long["date"] = pd.to_datetime(pm_long["date"], utc=True, format="mixed")
    pm_long["date"] = pm_long["date"].dt.round("H")

    # ── Resume: load already-fetched stations from existing CSV ──────────────
    if os.path.exists(output_csv):
        df_existing = pd.read_csv(output_csv)
        df_existing = df_existing.drop(
            columns=[c for c in df_existing.columns if "Unnamed" in c],
            errors="ignore"
        )
        done_stations = set(df_existing["stacja"].unique())
        print(f"  Resuming {output_csv}: {len(done_stations)} stations already saved.")
    else:
        df_existing   = pd.DataFrame()
        done_stations = set()
        print(f"  Starting fresh: {output_csv}")

    stations_todo = [s for s in df_meta.index if s not in done_stations]
    print(f"  Stations to fetch: {len(stations_todo)} / {len(df_meta.index)}")

    elevations = {}

    for i, station in enumerate(stations_todo, 1):
        lon = df_meta.at[station, "lon"]
        lat = df_meta.at[station, "lat"]
        print(f"  [{i}/{len(stations_todo)}] Fetching {resolution_label} "
              f"weather for {station} ({lat:.3f}, {lon:.3f})...")

        try:
            df_weather, elevation = fetch_fn(lon, lat)
        except Exception as e:
            print(f"    ERROR: {e} — skipping {station}")
            continue

        elevations[station] = elevation

        # Ensure date is a regular column
        if df_weather.index.dtype.kind == "M" or df_weather.index.name == "date":
            df_weather = df_weather.reset_index()

        df_weather["stacja"] = station

        # Align timezone
        df_weather["date"] = pd.to_datetime(df_weather["date"], utc=True)
        df_weather["date"] = df_weather["date"].dt.tz_convert("Europe/Warsaw")

        # Merge this station's weather with its PM2.5
        pm_station = pm_long[pm_long["stacja"] == station]
        df_station = pd.merge(pm_station, df_weather, on=["date", "stacja"], how="inner")

        # Append to existing and save immediately
        df_existing = pd.concat([df_existing, df_station], ignore_index=True)
        df_existing.to_csv(output_csv, index=False)

        time.sleep(API_SLEEP_S)

    # Store elevation back into metadata
    for station, elev in elevations.items():
        df_meta.at[station, "elevation"] = elev

    print(f"  → {output_csv} ({len(df_existing):,} rows, "
          f"{df_existing['stacja'].nunique()} stations)")
    return df_existing

'''
# Load metadata — index must be station codes, not row numbers.
# metadata5000best.csv has station codes in the first column ("stacja");
# we set it explicitly so the fetch loop addresses stations by code.
df_meta = pd.read_csv("metadata5000best.csv")
df_meta = df_meta.drop(
    columns=[c for c in df_meta.columns if "Unnamed" in c], errors="ignore"
)
# Station code column may be named 'stacja' or be the first unnamed index column
if "stacja" in df_meta.columns:
    df_meta = df_meta.set_index("stacja")
else:
    df_meta = df_meta.set_index(df_meta.columns[0])
    df_meta.index.name = "stacja"
print(f"Metadata loaded: {len(df_meta)} stations")
print(f"Index sample: {list(df_meta.index[:3])}")
'''

print("=== 24h weather ===")
df_full_24h = fetch_weather_all_stations(
    df_meta, df_pm25_24h, read_openmeteo,
    output_csv="stations_weather_2016_2019.csv",
    resolution_label="daily",
)

print("\n=== 1h weather ===")
df_full_1h = fetch_weather_all_stations(
    df_meta, df_pm25_1h, read_openmeteo_hour,
    output_csv="stations_weather_2016_2019_1h.csv",
    resolution_label="hourly",
)

# Save updated metadata with elevation
df_meta.to_csv("meta_elevation.csv")
print("\n→ meta_elevation.csv")


=== 24h weather ===
  Resuming stations_weather_2016_2019.csv: 8 stations already saved.
  Stations to fetch: 37 / 45
  [1/37] Fetching daily weather for LbLubSliwins (51.273, 22.552)...
Coordinates: 51.28295135498047°N 22.540538787841797°E
Elevation: 210.0 m asl
Timezone difference to GMT+0: 0s

Hourly data
                            date  boundary_layer_height
0     2016-01-01 00:00:00+00:00                  130.0
1     2016-01-01 01:00:00+00:00                  130.0
2     2016-01-01 02:00:00+00:00                  145.0
3     2016-01-01 03:00:00+00:00                  180.0
4     2016-01-01 04:00:00+00:00                  210.0
...                         ...                    ...
35083 2020-01-01 19:00:00+00:00                  760.0
35084 2020-01-01 20:00:00+00:00                  780.0
35085 2020-01-01 21:00:00+00:00                  765.0
35086 2020-01-01 22:00:00+00:00                  685.0
35087 2020-01-01 23:00:00+00:00                  680.0

[35088 rows x 2 columns]

Da

## 4. Copernicus GHSL raster features

Source: https://human-settlement.emergency.copernicus.eu/download.php?ds=bu  
Resolution: 100 m, reference year 2015, four tiles for Poland.

| File pattern | Product |
|---|---|
| `1v.tif … 4v.tif` | Built-up volume |
| `1p.tif … 4p.tif` | Population |
| `1s.tif … 4s.tif` | Built-up surface |

In [15]:
def merge_rasters(file_list: list, output_path: str) -> None:
    """
    Merge a list of GeoTIFF tiles into a single mosaic and save to output_path.
    Skips missing files with a warning.
    """
    src_files = []
    for fp in file_list:
        if not os.path.exists(fp):
            print(f"  [WARNING] {fp} not found — skipping")
            continue
        src_files.append(rasterio.open(fp))

    if not src_files:
        raise FileNotFoundError("No valid raster files found.")

    mosaic, transform = merge(src_files)
    crs = src_files[0].crs
    for src in src_files:
        src.close()

    out_meta = {
        "driver": "GTiff",
        "height": mosaic.shape[1],
        "width":  mosaic.shape[2],
        "count":  mosaic.shape[0],
        "dtype":  mosaic.dtype,
        "crs":    crs,
        "transform": transform,
    }
    with rasterio.open(output_path, "w", **out_meta) as dest:
        dest.write(mosaic)
    print(f"  Merged raster saved: {output_path}")


def raster_sum_in_buffer(
    lon: float, lat: float,
    mosaic_path: str,
    radius_m: int = BUFFER_RADIUS_M
) -> float:
    """
    Return the sum of raster values within a circular buffer around (lon, lat).
    Uses EPSG:2180 (Polish national metric CRS) for buffering.
    """
    gdf_point = gpd.GeoDataFrame({"geometry": [Point(lon, lat)]}, crs="EPSG:4326")
    buffer_m  = gdf_point.to_crs(epsg=2180).geometry.buffer(radius_m)

    with rasterio.open(mosaic_path) as src:
        buffer_proj = buffer_m.to_crs(src.crs)
        out_image, _ = mask(src, [buffer_proj.iloc[0]], crop=True)
        data = out_image[0].astype(float)
        if src.nodata is not None:
            data[data == src.nodata] = np.nan

    return float(np.nansum(data))


# Merge raster tiles
RASTER_PRODUCTS = {
    "population.tif": [f"{i}p.tif" for i in range(1, 5)],
    "volume.tif":     [f"{i}v.tif" for i in range(1, 5)],
    "surface.tif":    [f"{i}s.tif" for i in range(1, 5)],
}

for output, tiles in RASTER_PRODUCTS.items():
    if not os.path.exists(output):
        print(f"Merging {output}...")
        merge_rasters(tiles, output)
    else:
        print(f"  {output} already exists — skipping")

  population.tif already exists — skipping
  volume.tif already exists — skipping
  surface.tif already exists — skipping


In [16]:
# Extract raster features per station
df_meta = pd.read_csv("meta_elevation.csv", index_col=0)
df_meta.index.name = "stacja"

populations = []
volumes     = []
surfaces    = []

for i, (station, row) in enumerate(df_meta.iterrows(), 1):
    print(f"  [{i}/{len(df_meta)}] {station}")
    populations.append(raster_sum_in_buffer(row["lon"], row["lat"], "population.tif"))
    volumes.append(    raster_sum_in_buffer(row["lon"], row["lat"], "volume.tif"))
    surfaces.append(   raster_sum_in_buffer(row["lon"], row["lat"], "surface.tif"))

df_meta["population"]      = populations
df_meta["built_up_volume"]  = volumes
df_meta["built_up_surface"] = surfaces

df_meta.to_csv("metaall.csv")
print(f"\n→ metaall.csv ({len(df_meta)} stations)")
df_meta.head()

  [1/45] DsOsieczow21
  [2/45] DsWalbrzWyso
  [3/45] DsWrocNaGrob
  [4/45] DsZgorBohGet
  [5/45] KpBydFieldor
  [6/45] KpGrudSienki
  [7/45] KpToruDziewu
  [8/45] LbBiaPodOrze
  [9/45] LbLubSliwins
  [10/45] LbZamoHrubie
  [11/45] LdLodzCzerni
  [12/45] LdLodzLegion
  [13/45] LdPioTrKraPr
  [14/45] LuGorzPilsud
  [15/45] LuZielKrotka
  [16/45] MpBochKonfed
  [17/45] MpKrakBujaka
  [18/45] MpNoSaczNadb
  [19/45] MpTarBitStud
  [20/45] MpTrzebOsZWM
  [21/45] MpZakopaSien
  [22/45] MzRadHallera
  [23/45] MzWarWokalna
  [24/45] OpKluczMicki
  [25/45] OpOpoleOsAKr
  [26/45] PdBialWarsza
  [27/45] PdLomSikorsk
  [28/45] PkJasloSikor
  [29/45] PkNiskoSzkla
  [30/45] PkRzeszRejta
  [31/45] SlBielSterni
  [32/45] SlCzestoZana
  [33/45] SlGliwicMewy
  [34/45] SlGodGliniki
  [35/45] SlKatoKossut
  [36/45] SlTarnoLitew
  [37/45] SlZlotPotLes
  [38/45] SlZorySikor2
  [39/45] WmElbBazynsk
  [40/45] WmOlsPuszkin
  [41/45] WmOstrPilsud
  [42/45] WmPuszczaBor
  [43/45] ZpKoszSpasow
  [44/45] ZpSzczAndr

,lat,lon,nearest_road_dist_m,total_road_length_km,building_count,elevation,population,built_up_volume,built_up_surface
stacja,,,,,,,,,
DsOsieczow21,51.317630,15.431719,40.258760,148.954010,2334.0,179.0,2240.586464,859436.0,343715.0
DsWalbrzWyso,50.768729,16.269677,170.100903,817.813995,15897.0,441.0,102796.665402,22788216.0,4756994.0
DsWrocNaGrob,51.103456,17.059225,97.176639,1518.020860,60249.0,123.0,368776.490393,171569591.0,14934688.0
DsZgorBohGet,51.150391,15.008175,138.987771,988.524194,28363.0,213.0,83586.284188,34522676.0,6462650.0
KpBydFieldor,53.151452,18.132062,22.380118,711.014879,8995.0,54.0,56629.378719,31829943.0,4386547.0


## 5. Final merge — all_data.csv / all_data_1h.csv

In [17]:
df_meta = pd.read_csv("metaall.csv", index_col=0)
# Drop any leftover index columns from intermediate saves
df_meta = df_meta.drop(
    columns=[c for c in df_meta.columns if "Unnamed" in c], errors="ignore"
)
df_meta.index.name = "stacja"
df_meta = df_meta.reset_index()   # stacja becomes a regular column for merge

for weather_csv, output_csv in [
    ("stations_weather_2016_2019.csv",    "all_data.csv"),
    ("stations_weather_2016_2019_1h.csv", "all_data_1h.csv"),
]:
    df_weather = pd.read_csv(weather_csv)
    df_weather = df_weather.drop(
        columns=[c for c in df_weather.columns if "Unnamed" in c], errors="ignore"
    )
    df_all = pd.merge(df_weather, df_meta, on="stacja", how="inner")
    df_all.to_csv(output_csv, index=False)
    print(f"→ {output_csv}: {df_all.shape[0]:,} rows × {df_all.shape[1]} cols")

print("\nDone.")

→ all_data.csv: 65,745 rows × 21 cols
→ all_data_1h.csv: 280,504 rows × 21 cols

Done.


## 6. Summary

In [18]:
df_all = pd.read_csv("all_data.csv")

print("=" * 60)
print("DATASET SUMMARY (all_data.csv)")
print("=" * 60)
print(f"  Rows          : {df_all.shape[0]:,}")
print(f"  Columns       : {df_all.shape[1]}")
print(f"  Stations      : {df_all['stacja'].nunique()}")
print(f"  Date range    : {df_all['date'].min()} – {df_all['date'].max()}")
print(f"  PM2.5 missing : {df_all['pm25'].isna().sum():,} ({100*df_all['pm25'].isna().mean():.1f}%)")
print()
print("Columns:")
for col in df_all.columns:
    print(f"  {col}")

DATASET SUMMARY (all_data.csv)
  Rows          : 65,745
  Columns       : 21
  Stations      : 45
  Date range    : 2016-01-01 00:00:00+00:00 – 2019-12-31 00:00:00+00:00
  PM2.5 missing : 2,515 (3.8%)

Columns:
  date
  stacja
  pm25
  temperature_2m_min
  wind_speed_10m_max
  wind_direction_10m_dominant
  cloud_cover_mean
  surface_pressure_mean
  dew_point_2m_mean
  relative_humidity_2m_min
  wind_speed_10m_mean
  boundary_layer_height
  lat
  lon
  nearest_road_dist_m
  total_road_length_km
  building_count
  elevation
  population
  built_up_volume
  built_up_surface
